<a href="https://colab.research.google.com/github/LeonardooAlves/WM9G1-BDAI/blob/main/Week%201/2_Big_Data_Ingestion_Intro_to_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Big Data Ingestion: Working with APIs

**Course:** MSc Engineering Business Management - Big Data Analytics for Industry

---

## Learning Objectives

By the end of this tutorial, you will understand:
1. **How APIs work** - Request/response cycle, HTTP methods, authentication
2. **Data formats** - JSON and how to parse them

# Extracting Data from APIs: A Step-by-Step Tutorial

## Introduction

In previous sessions, we discussed the fundamental concepts of **Application Programming Interfaces (APIs)** and how they enable different software applications to communicate with each other. We also covered the basics of how APIs work in the overview document available on GitHub ([Understanding APIs.MD](https://github.com/LeonardooAlves/WM9G1-BDAI/blob/2db25524aa125f22e01700e9d4ab6f696409f443/Week%201/1_Understanding_APIs.md)). If you need a refresher on what APIs are and how they function, please review that material before proceeding.

## What We'll Build Today

In this tutorial, we will work through a practical example of extracting and combining data from **three different APIs**. This will demonstrate how multiple APIs can work together to create a more complex application. While our example is somewhat playful, it illustrates important real-world concepts about API integration.

### The APIs We'll Use

We will interact with the following three public APIs, each serving a different purpose:

1. **RandomUser API** ([documentation](https://randomuser.me/documentation))
   - **Purpose:** Generates fake/random user profiles with realistic data
   - **What we'll get:** Names, locations, contact details, and other user information
   - **Why it's useful:** This is similar to test data generation in software development

2. **Agify.io API** ([documentation](https://publicapi.dev/agify-io-api))
   - **Purpose:** Predicts the age of a person based on their first name
   - **What we'll get:** An estimated age and confidence count
   - **How it works:** Uses statistical analysis of name databases across cultures

3. **Nationalize.io API** ([documentation](https://nationalize.io/documentation))
   - **Purpose:** Predicts the nationality of a person based on their first name
   - **What we'll get:** Likely country codes with probability scores
   - **How it works:** Analyzes name patterns and their geographical distributions

### Our Workflow

Here's what we'll accomplish step by step:

1. **First**, we'll generate a random user profile using the RandomUser API
2. **Second**, we'll extract the user's first name and use it to predict their age with the Agify.io API
3. **Third**, we'll use the same first name to predict their nationality with the Nationalize.io API
4. **Finally**, we'll automate this entire process by creating reusable functions

### Why This Matters

While this specific task might seem trivial, the skills you'll learn are directly applicable to real business scenarios:

- **Data enrichment:** Companies often combine data from multiple sources to create comprehensive customer profiles
- **API integration:** Modern applications rarely rely on a single data source; they integrate multiple APIs
- **Automation:** Once you understand how APIs work individually, you can chain them together to create powerful automated workflows

### A Note About These APIs

An important advantage of the APIs we're using today is that they **require no authentication keys** or registration. This makes them ideal for learning because:
- You can start experimenting immediately without signing up
- There's no risk of exposing sensitive credentials
- The code examples are simpler and more focused on core concepts

In real-world applications, most APIs will require authentication (API keys, OAuth tokens, etc.), but the fundamental request-response pattern we'll learn here remains the same.

Let's begin by setting up our environment!

---

## Step 0: Setting Up Our Environment

Before we can start making API calls, we need to import the necessary Python libraries. Think of libraries as toolboxes that contain pre-written code to help us accomplish specific tasks.

### The Libraries We Need

We'll be using two essential libraries for working with APIs:

1. **`requests`**
   - **Purpose:** This library handles HTTP requests and responses
   - **What it does:** It allows Python to communicate with web servers and APIs over the internet
   - **Why we need it:** Making API calls is essentially sending HTTP requests to remote servers

2. **`json`**
   - **Purpose:** This library works with JSON (JavaScript Object Notation) data format
   - **What it does:** It converts JSON text into Python dictionaries and vice versa
   - **Why we need it:** Most modern APIs return data in JSON format, which we need to parse and work with

### Understanding JSON

JSON is a lightweight data format that's easy for both humans and computers to read. It looks very similar to Python dictionaries. For example:

```json
{
  "name": "John",
  "age": 30,
  "city": "London"
}
```

Let's import our libraries and set up the API endpoint URL:

In [1]:
# Import the libraries we need for API communication
import json      # For handling JSON data format
import requests  # For making HTTP requests to APIs

# Define the base URL for the RandomUser API
# This is called an "endpoint" - it's the web address where we send our requests
random_user_url = "https://randomuser.me/api/"

print("✓ Libraries imported successfully")
print(f"✓ RandomUser API endpoint ready: {random_user_url}")

✓ Libraries imported successfully
✓ RandomUser API endpoint ready: https://randomuser.me/api/


**What just happened?**

- We imported the `requests` and `json` libraries, making their functions available to use
- We created a variable called `random_user_url` that stores the API endpoint address
- The endpoint URL is the base address we'll use to communicate with the RandomUser API

**Important concept:** An API endpoint is like a post office address. Just as you need the correct address to send a letter, you need the correct URL to send an API request. The API "lives" at this address and waits for requests to arrive.

---

## Step 1: Getting a Random User

Now let's make our first API call! We'll request a random user profile from the RandomUser API.

### Understanding the Request-Response Cycle

When we make an API call, here's what happens:

1. **Your code sends a REQUEST** to the API server (like asking a question)
2. **The API server processes** your request (like thinking about the answer)
3. **The server sends back a RESPONSE** with the data (like giving you the answer)

This is similar to ordering food at a restaurant:
- You make a request (order food)
- The kitchen processes it (cooks your meal)
- You receive a response (your food arrives)

### Making the Request

We'll use the `requests.get()` function to send a **GET request** to the API. GET is an HTTP method that means "retrieve data" (as opposed to POST which means "send data").

In [2]:
# Make a GET request to the RandomUser API
# The requests.get() function sends an HTTP GET request to the specified URL
response = requests.get(random_user_url)

# Let's examine what we got back
print("API Response Object:")
print(response)
print("\nResponse Status Code:", response.status_code)
print("Response Type:", type(response))

API Response Object:
<Response [200]>

Response Status Code: 200
Response Type: <class 'requests.models.Response'>


**Understanding what we received:**

- **The response object:** This is not the actual data yet - it's a Python object that contains the server's response
- **Status code 200:** This is HTTP code for "Success" - the request worked perfectly
  - Other common codes: 404 (Not Found), 401 (Unauthorized), 500 (Server Error)
- **Response type:** It's a `requests.Response` object, which has several useful properties

### Viewing the Raw Response Data

The response object contains the data, but it's stored as raw bytes. Let's look at it:

In [3]:
# View the raw content of the response
# The .content attribute contains the response data as bytes (raw binary data)
print("Raw Response Content:")
print(response.content)
print("\nContent Type:", type(response.content))

Raw Response Content:
b'{"results":[{"gender":"male","name":{"title":"Mr","first":"Gerjon","last":"Postmus"},"location":{"street":{"number":2719,"name":"Irenestraat"},"city":"Sappemeer","state":"Groningen","country":"Netherlands","postcode":"0094 DH","coordinates":{"latitude":"36.5657","longitude":"-56.2716"},"timezone":{"offset":"+6:00","description":"Almaty, Dhaka, Colombo"}},"email":"gerjon.postmus@example.com","login":{"uuid":"58c4cb53-ae54-439e-a720-b3d6ee02d64f","username":"blackbear755","password":"iverson","salt":"Q6uqmL56","md5":"041db196c3cd53193e5cc5d7d43122ba","sha1":"3143a5b6d5e6d0c357c3d26f88d0200f6d448cec","sha256":"75f1ea49b8c215044ab1804f8150ba0586055a6378ac8ab6c3da86960c29ee55"},"dob":{"date":"1955-09-14T03:13:29.659Z","age":70},"registered":{"date":"2018-12-24T16:35:56.449Z","age":7},"phone":"(098) 0184292","cell":"(06) 92109349","id":{"name":"BSN","value":"52853714"},"picture":{"large":"https://randomuser.me/api/portraits/men/41.jpg","medium":"https://randomuser.me/

**What are we looking at?**

You can see the data starts with `b'` which indicates **bytes** - the raw binary format that computers use to transmit data over the internet. While this contains all the information we need, it's difficult to work with in this format.

The data is structured as JSON, but it's currently just a long string of text. We need to convert it into a format Python can easily manipulate.

### Converting to JSON

The `response` object has a convenient method called `.json()` that automatically:
1. Takes the raw bytes
2. Decodes them into text
3. Parses the JSON structure
4. Converts it into a Python dictionary

Let's do that conversion:

In [4]:
# Convert the JSON response into a Python dictionary
# The .json() method automatically handles all the parsing for us
user_data = response.json()

# Display the nicely formatted data
print("Converted User Data (as Python dictionary):")
print(user_data)
print("\nData Type:", type(user_data))

Converted User Data (as Python dictionary):
{'results': [{'gender': 'male', 'name': {'title': 'Mr', 'first': 'Gerjon', 'last': 'Postmus'}, 'location': {'street': {'number': 2719, 'name': 'Irenestraat'}, 'city': 'Sappemeer', 'state': 'Groningen', 'country': 'Netherlands', 'postcode': '0094 DH', 'coordinates': {'latitude': '36.5657', 'longitude': '-56.2716'}, 'timezone': {'offset': '+6:00', 'description': 'Almaty, Dhaka, Colombo'}}, 'email': 'gerjon.postmus@example.com', 'login': {'uuid': '58c4cb53-ae54-439e-a720-b3d6ee02d64f', 'username': 'blackbear755', 'password': 'iverson', 'salt': 'Q6uqmL56', 'md5': '041db196c3cd53193e5cc5d7d43122ba', 'sha1': '3143a5b6d5e6d0c357c3d26f88d0200f6d448cec', 'sha256': '75f1ea49b8c215044ab1804f8150ba0586055a6378ac8ab6c3da86960c29ee55'}, 'dob': {'date': '1955-09-14T03:13:29.659Z', 'age': 70}, 'registered': {'date': '2018-12-24T16:35:56.449Z', 'age': 7}, 'phone': '(098) 0184292', 'cell': '(06) 92109349', 'id': {'name': 'BSN', 'value': '52853714'}, 'picture':

In [5]:
#Alternatively, you can print it like this
user_data

{'results': [{'gender': 'male',
   'name': {'title': 'Mr', 'first': 'Gerjon', 'last': 'Postmus'},
   'location': {'street': {'number': 2719, 'name': 'Irenestraat'},
    'city': 'Sappemeer',
    'state': 'Groningen',
    'country': 'Netherlands',
    'postcode': '0094 DH',
    'coordinates': {'latitude': '36.5657', 'longitude': '-56.2716'},
    'timezone': {'offset': '+6:00', 'description': 'Almaty, Dhaka, Colombo'}},
   'email': 'gerjon.postmus@example.com',
   'login': {'uuid': '58c4cb53-ae54-439e-a720-b3d6ee02d64f',
    'username': 'blackbear755',
    'password': 'iverson',
    'salt': 'Q6uqmL56',
    'md5': '041db196c3cd53193e5cc5d7d43122ba',
    'sha1': '3143a5b6d5e6d0c357c3d26f88d0200f6d448cec',
    'sha256': '75f1ea49b8c215044ab1804f8150ba0586055a6378ac8ab6c3da86960c29ee55'},
   'dob': {'date': '1955-09-14T03:13:29.659Z', 'age': 70},
   'registered': {'date': '2018-12-24T16:35:56.449Z', 'age': 7},
   'phone': '(098) 0184292',
   'cell': '(06) 92109349',
   'id': {'name': 'BSN', '

**Much better!** Now we can see the data is formatted as a Python dictionary with clear key-value pairs. This is the same data as before, but now it's in a structure we can easily navigate and extract information from.

### Understanding the Data Structure

Let's examine the structure of what the API returned. Notice the data has two main keys:

1. **`'results'`**: Contains an array (list) of user objects - in this case, just one user
2. **`'info'`**: Contains metadata about the API response (like version number, how many results, etc.)

The user data itself is nested inside `results[0]` (the first item in the results list). Let's extract just the user information:

In [6]:
# Extract the actual user information from the results array
# user_data['results'] is a list, and [0] gets the first (and only) item
user = user_data['results'][0]

print("User Information:")
print(user)
print("\nData Type:", type(user))

User Information:
{'gender': 'male', 'name': {'title': 'Mr', 'first': 'Gerjon', 'last': 'Postmus'}, 'location': {'street': {'number': 2719, 'name': 'Irenestraat'}, 'city': 'Sappemeer', 'state': 'Groningen', 'country': 'Netherlands', 'postcode': '0094 DH', 'coordinates': {'latitude': '36.5657', 'longitude': '-56.2716'}, 'timezone': {'offset': '+6:00', 'description': 'Almaty, Dhaka, Colombo'}}, 'email': 'gerjon.postmus@example.com', 'login': {'uuid': '58c4cb53-ae54-439e-a720-b3d6ee02d64f', 'username': 'blackbear755', 'password': 'iverson', 'salt': 'Q6uqmL56', 'md5': '041db196c3cd53193e5cc5d7d43122ba', 'sha1': '3143a5b6d5e6d0c357c3d26f88d0200f6d448cec', 'sha256': '75f1ea49b8c215044ab1804f8150ba0586055a6378ac8ab6c3da86960c29ee55'}, 'dob': {'date': '1955-09-14T03:13:29.659Z', 'age': 70}, 'registered': {'date': '2018-12-24T16:35:56.449Z', 'age': 7}, 'phone': '(098) 0184292', 'cell': '(06) 92109349', 'id': {'name': 'BSN', 'value': '52853714'}, 'picture': {'large': 'https://randomuser.me/api/p

### Extracting Specific Information

Now that we have the user data as a dictionary, we can access specific pieces of information using **keys**. This works just like accessing items in a Python dictionary.

The user data contains many nested dictionaries. For example, the name information is stored like this:

```python
user['name'] = {
  'title': 'Mr',
  'first': 'John',
  'last': 'Doe'
}
```

To get the first name, we need to:
1. Access the 'name' dictionary: `user['name']`
2. Then access the 'first' key within that: `user['name']['first']`

Let's extract the first name from our random user:

In [7]:
# Extract the first name from the nested name dictionary
# This uses "chain indexing" - first accessing 'name', then 'first' within it
first_name = user['name']['first']

print(f"Random User's First Name: {first_name}")
print(f"Type: {type(first_name)}")

# Let's also extract some other useful information while we're at it
last_name = user['name']['last']
actual_age = user['dob']['age']  # The actual age from the user profile
actual_country = user['location']['country']

print(f"\nComplete User Information:")
print(f"  Name: {first_name} {last_name}")
print(f"  Actual Age: {actual_age}")
print(f"  Actual Country: {actual_country}")

Random User's First Name: Gerjon
Type: <class 'str'>

Complete User Information:
  Name: Gerjon Postmus
  Actual Age: 70
  Actual Country: Netherlands


**Perfect!** We now have:

- The user's first name stored in the `first_name` variable
- Their actual age and country (which we'll compare to our predictions later)

### What We've Learned So Far

In this first step, we've learned how to:

1. ✓ Send a GET request to an API using `requests.get()`
2. ✓ Check the status code to verify the request succeeded
3. ✓ Convert JSON response data into a Python dictionary using `.json()`
4. ✓ Navigate nested dictionary structures to extract specific information
5. ✓ Access nested data using chain indexing like `user['name']['first']`

**Important takeaway:** The pattern we just used (make request → check status → parse JSON → extract data) is the fundamental workflow for working with any REST API.

Now we're ready to use this first name to query our second API!

---

## Step 2: Predicting Age with Agify.io

Now that we have a random user's first name, let's use it to predict their age using the **Agify.io API**.

### How Agify.io Works

The Agify.io API analyzes the first name you provide and returns:
- A predicted age based on statistical data
- A count indicating how many people with that name were in their database
- The higher the count, the more confident the prediction

For example, the name "William" might predict an older age because it was more common in previous generations, while "Jayden" might predict a younger age because it's a more recent name trend.

### Understanding API Parameters

Unlike our RandomUser API call (which took no parameters), Agify.io requires us to tell it **which name** we want to analyze. We do this by adding **query parameters** to the URL.

Query parameters are added to a URL after a question mark `?` and follow this pattern:
```
https://api.example.com/endpoint?parameter_name=value
```

For Agify.io, the parameter is called `name`, so our URL will look like:
```
https://api.agify.io?name=John
```

If we had multiple parameters, we'd separate them with `&`:
```
https://api.agify.io?name=John&country_id=US
```

### Constructing the API Request

Let's build the URL for the Agify API call. We'll create the base URL, then add the name parameter:

In [8]:
# Define the base URL for the Agify.io API
agify_base_url = "https://api.agify.io"

# Construct the complete URL by adding the name parameter
# We use string concatenation to build: base_url + ?name= + the actual name
agify_url = f"{agify_base_url}?name={first_name}"

print(f"Agify API Base URL: {agify_base_url}")
print(f"Name to analyze: {first_name}")
print(f"\nComplete API URL: {agify_url}")
print("\nThis URL tells Agify: 'Please predict the age for this name'")

Agify API Base URL: https://api.agify.io
Name to analyze: Gerjon

Complete API URL: https://api.agify.io?name=Gerjon

This URL tells Agify: 'Please predict the age for this name'


**Understanding the URL we created:**

- `https://api.agify.io` - The API's base endpoint
- `?` - Indicates that parameters are coming next
- `name=` - The parameter name (this is defined by the API's documentation)
- `{first_name}` - The actual value we're sending (the name we got from RandomUser)

### Making the API Call

Now let's send the request to Agify.io. We'll use the same pattern as before: make the request, check the status, and examine the response.

In [9]:
# Send a GET request to the Agify API
agify_response = requests.get(agify_url)

# Check if the request was successful
print(f"Response Status Code: {agify_response.status_code}")

if agify_response.status_code == 200:
    print("✓ Request successful!\n")
else:
    print("✗ Request failed!\n")

# Look at the raw response
print("Raw Response Content:")
print(agify_response.content)

Response Status Code: 200
✓ Request successful!

Raw Response Content:
b'{"count":9,"name":"Gerjon","age":44}'


### Parsing the Age Prediction

Just like before, we need to convert the JSON response into a Python dictionary so we can work with it easily:

In [10]:
# Convert the JSON response to a Python dictionary
agify_data = agify_response.json()

print("Agify API Response (as dictionary):")
print(agify_data)
print(f"\nData Type: {type(agify_data)}")

Agify API Response (as dictionary):
{'count': 9, 'name': 'Gerjon', 'age': 44}

Data Type: <class 'dict'>


**Understanding the Agify response:**

The response contains three pieces of information:

1. **`name`**: The name that was analyzed (echoed back to us)
2. **`age`**: The predicted age
3. **`count`**: How many people with this name were in their database
   - Higher count = more reliable prediction
   - Low count = less reliable (might be an uncommon name)

Notice this response has a **much simpler structure** than the RandomUser response. It's just a flat dictionary with three keys - no nesting required!

### Extracting the Predicted Age

Let's extract the age prediction and compare it to the actual age:

In [11]:
# Extract the predicted age from the response
# Since this is a simple dictionary, we just need one key
predicted_age = agify_data['age']
confidence_count = agify_data['count']

print("=" * 60)
print("AGE PREDICTION RESULTS")
print("=" * 60)
print(f"Name analyzed: {first_name}")
print(f"Predicted age: {predicted_age}")
print(f"Confidence (database count): {confidence_count:,}")
print(f"\nActual age: {actual_age}")
print(f"Difference: {abs(predicted_age - actual_age) if predicted_age else 'N/A'} years")
print("=" * 60)

AGE PREDICTION RESULTS
Name analyzed: Gerjon
Predicted age: 44
Confidence (database count): 9

Actual age: 70
Difference: 26 years


**Interpreting the results:**

- If `predicted_age` is `None` (null), it means the API doesn't have enough data about this name to make a prediction
- The difference between predicted and actual age shows how accurate the API's guess was
- The count tells us how confident we should be in this prediction

### What We've Learned

In this step, we learned how to:

1. ✓ Add query parameters to API URLs using the `?parameter=value` syntax
2. ✓ Construct dynamic URLs by combining base URLs with variable values
3. ✓ Work with simpler, flat JSON response structures
4. ✓ Handle potential null/None values in API responses
5. ✓ Extract and compare data from multiple sources (RandomUser vs Agify)

**Key concept:** Different APIs have different data structures. RandomUser returned complex nested data, while Agify returned a simple flat structure. Always check the API documentation to understand what format to expect!

Now let's move on to predicting nationality!

---

## Step 3: Predicting Nationality with Nationalize.io

For our final individual API call, let's predict the nationality of our random user based on their first name using the **Nationalize.io API**.

### How Nationalize.io Works

The Nationalize.io API analyzes first names and returns:
- A list of likely countries where this name is common
- Each country is represented by its **ISO country code** (like 'US', 'GB', 'FR')
- A **probability score** for each country (indicating confidence)

For example:
- The name "Mohammed" might return high probabilities for countries like Saudi Arabia (SA), Egypt (EG), or Pakistan (PK)
- The name "Sean" might return high probabilities for Ireland (IE), United States (US), or United Kingdom (GB)

### Understanding the Response Format

Unlike Agify which returns a single age prediction, Nationalize returns **multiple possibilities** ranked by probability. This is more realistic because many names are common across multiple countries.

### Building the Request

Just like with Agify, we need to add a `name` parameter to the URL. The process is identical:

In [12]:
# Define the base URL for the Nationalize.io API
nationalize_base_url = "https://api.nationalize.io"

# Construct the complete URL with the name parameter
nationalize_url = f"{nationalize_base_url}?name={first_name}"

print(f"Nationalize API Base URL: {nationalize_base_url}")
print(f"Name to analyze: {first_name}")
print(f"\nComplete API URL: {nationalize_url}")
print("\nThis URL tells Nationalize: 'Please predict the nationality for this name'")

Nationalize API Base URL: https://api.nationalize.io
Name to analyze: Gerjon

Complete API URL: https://api.nationalize.io?name=Gerjon

This URL tells Nationalize: 'Please predict the nationality for this name'


**Notice the pattern:** We're using the exact same approach as with Agify - construct base URL, add parameter, make request. This is a common pattern across most REST APIs.

### Making the API Call

In [13]:
# Send a GET request to the Nationalize API
nationalize_response = requests.get(nationalize_url)

# Check the status
print(f"Response Status Code: {nationalize_response.status_code}")

if nationalize_response.status_code == 200:
    print("✓ Request successful!\n")
else:
    print("✗ Request failed!\n")

# Look at the raw response
print("Raw Response Content:")
print(nationalize_response.content)

Response Status Code: 200
✓ Request successful!

Raw Response Content:
b'{"count":2,"name":"Gerjon","country":[{"country_id":"AL","probability":0.7142857142857143}]}'


### Parsing the Nationality Predictions

Let's convert the response to a dictionary and examine its structure:

In [14]:
# Convert the JSON response to a Python dictionary
nationalize_data = nationalize_response.json()

print("Nationalize API Response (as dictionary):")
print(nationalize_data)
print(f"\nData Type: {type(nationalize_data)}")

Nationalize API Response (as dictionary):
{'count': 2, 'name': 'Gerjon', 'country': [{'country_id': 'AL', 'probability': 0.7142857142857143}]}

Data Type: <class 'dict'>


**Understanding the Nationalize response structure:**

The response contains:

1. **`name`**: The name that was analyzed
2. **`country`**: An array (list) of predictions, each containing:
   - `country_id`: The ISO country code (e.g., 'US', 'GB', 'FR')
   - `probability`: A decimal between 0 and 1 indicating confidence

**Key difference from Agify:** Instead of a single value, we get a **list of dictionaries**. Each item in the list represents a different country prediction.

For example:
```python
{
  'name': 'John',
  'country': [
    {'country_id': 'US', 'probability': 0.35},
    {'country_id': 'GB', 'probability': 0.25},
    {'country_id': 'CA', 'probability': 0.15}
  ]
}
```

### Extracting the Top Prediction

The countries are returned in order of probability (highest first), so the first country in the list is the API's best guess. Let's extract it:

In [15]:
# Extract the country predictions list
country_predictions = nationalize_data['country']

print(f"Number of country predictions: {len(country_predictions)}")
print(f"\nAll predictions:")
print(country_predictions)

# Get the top prediction (first item in the list)
# We use [0] to get the first dictionary from the list
if len(country_predictions) > 0:
    top_prediction = country_predictions[0]
    predicted_country_code = top_prediction['country_id']
    prediction_probability = top_prediction['probability']

    print(f"\nTop Prediction:")
    print(f"  Country Code: {predicted_country_code}")
    print(f"  Confidence: {prediction_probability:.2%}")  # Format as percentage
else:
    predicted_country_code = None
    prediction_probability = 0
    print("\nNo predictions available for this name")

Number of country predictions: 1

All predictions:
[{'country_id': 'AL', 'probability': 0.7142857142857143}]

Top Prediction:
  Country Code: AL
  Confidence: 71.43%


### Displaying All Predictions

While we're primarily interested in the top prediction, it's interesting to see all the possibilities. Let's display them in a more readable format:

In [16]:
# Display all country predictions in a formatted way
print("=" * 60)
print("NATIONALITY PREDICTION RESULTS")
print("=" * 60)
print(f"Name analyzed: {first_name}\n")

if len(country_predictions) > 0:
    print("All Predictions (ranked by probability):\n")

    # Loop through each prediction and display it
    for i, prediction in enumerate(country_predictions, 1):
        country_code = prediction['country_id']
        probability = prediction['probability']
        print(f"  {i}. {country_code}: {probability:.2%}")

    print(f"\n✓ Best guess: {predicted_country_code} ({prediction_probability:.2%} confidence)")
else:
    print("  No predictions available (name not in database)")

print(f"\nActual country: {actual_country}")
print("=" * 60)

NATIONALITY PREDICTION RESULTS
Name analyzed: Gerjon

All Predictions (ranked by probability):

  1. AL: 71.43%

✓ Best guess: AL (71.43% confidence)

Actual country: Netherlands


**Understanding probability scores:**

- Probabilities are decimals between 0 and 1 (we converted to percentages for readability)
- They represent how common this name is in each country
- The probabilities across all countries sum to approximately 1.0 (100%)
- Higher probability = API is more confident about that country

**Important note about country codes:**

The API returns ISO 3166-1 alpha-2 country codes:
- US = United States
- GB = United Kingdom (Great Britain)
- FR = France
- DE = Germany
- etc.

You can find a complete list of country codes [here](https://en.wikipedia.org/wiki/ISO_3166-1_alpha-2).

### What We've Learned

In this step, we learned how to:

1. ✓ Work with API responses that contain **lists** of data (arrays)
2. ✓ Extract the first item from a list using `[0]` indexing
3. ✓ Access nested data within lists of dictionaries: `data['country'][0]['country_id']`
4. ✓ Loop through all items in an API response to display multiple results
5. ✓ Handle cases where APIs return empty lists (no data available)
6. ✓ Format probability values as percentages for better readability

**Key insight:** APIs can return different data structures:
- Simple values (Agify: single age)
- Lists of objects (Nationalize: multiple country predictions)
- Complex nested structures (RandomUser: deeply nested user profile)

Understanding these structures is crucial for extracting the data you need!

---

## Summary of Individual API Calls

We've now successfully:

1. ✓ Generated a random user profile from RandomUser API
2. ✓ Extracted their first name from the nested JSON structure
3. ✓ Predicted their age using Agify.io API
4. ✓ Predicted their nationality using Nationalize.io API
5. ✓ Compared predictions to actual values

**We did all of this without using functions** - writing out each step explicitly to understand the process. Now that we understand how each API works individually, we're ready to automate this entire workflow using functions!

---

## Step 4: Automating the Process with Functions

Now that we understand how each API works individually, let's create **reusable functions** to automate the entire workflow. This demonstrates a key programming principle: once you understand how something works, you can package it into a function for easy reuse.

### Why Create Functions?

Functions provide several benefits:

1. **Reusability:** Write the code once, use it many times
2. **Organization:** Group related code together logically
3. **Maintainability:** If something needs to change, you update it in one place
4. **Abstraction:** Hide complex details behind a simple interface

We'll create four functions:
1. `get_random_user()` - Generates a random user
2. `predict_age()` - Predicts age from a name
3. `predict_nationality()` - Predicts nationality from a name
4. `analyze_user()` - Combines all three into one workflow

### Function 1: Get Random User

This function will encapsulate everything we did in Step 1:

In [17]:
def get_random_user():
    """
    Retrieves a random user profile from the RandomUser API.

    Returns:
        dict: A dictionary containing the user's information with these keys:
            - 'first_name': str, the user's first name
            - 'last_name': str, the user's last name
            - 'age': int, the user's actual age
            - 'country': str, the user's actual country
            - 'full_data': dict, the complete API response for reference
    """
    # Make the API request
    url = "https://randomuser.me/api/"
    response = requests.get(url)

    # Convert to JSON and extract the user data
    data = response.json()
    user = data['results'][0]

    # Extract and return the relevant information in a clean format
    return {
        'first_name': user['name']['first'],
        'last_name': user['name']['last'],
        'age': user['dob']['age'],
        'country': user['location']['country'],
        'full_data': user  # Keep the full data in case we need other fields
    }

# Test the function
print("Testing get_random_user() function...\n")
test_user = get_random_user()
print("Generated user:")
print(f"  Name: {test_user['first_name']} {test_user['last_name']}")
print(f"  Age: {test_user['age']}")
print(f"  Country: {test_user['country']}")

Testing get_random_user() function...

Generated user:
  Name: Simon Jørgensen
  Age: 71
  Country: Denmark


**What this function does:**

- Takes **no parameters** (inputs) because RandomUser doesn't need any
- Makes the API call internally
- Extracts the important fields
- **Returns** a clean dictionary with just the data we care about
- Includes a **docstring** (the text in triple quotes) explaining what it does

**Key benefit:** Instead of writing 6+ lines of code each time we want a random user, we can now just call `get_random_user()` and it handles everything!

### Function 2: Predict Age

This function encapsulates our Agify.io API call from Step 2:

In [18]:
def predict_age(first_name):
    """
    Predicts the age of a person based on their first name using Agify.io API.

    Parameters:
        first_name (str): The first name to analyze

    Returns:
        dict: A dictionary containing:
            - 'name': str, the name that was analyzed
            - 'predicted_age': int or None, the predicted age
            - 'count': int, number of data points used for prediction
    """
    # Construct the API URL with the name parameter
    url = f"https://api.agify.io?name={first_name}"

    # Make the request and parse the response
    response = requests.get(url)
    data = response.json()

    # Return the prediction data
    return {
        'name': data['name'],
        'predicted_age': data['age'],  # Could be None if name not in database
        'count': data['count']
    }

# Test the function
print("Testing predict_age() function...\n")
age_result = predict_age(test_user['first_name'])
print(f"Age prediction for '{age_result['name']}':")
print(f"  Predicted: {age_result['predicted_age']}")
print(f"  Confidence: {age_result['count']:,} data points")
print(f"  Actual age: {test_user['age']}")

Testing predict_age() function...

Age prediction for 'Simon':
  Predicted: 53
  Confidence: 69,340 data points
  Actual age: 71


**What this function does:**

- Takes **one parameter:** the `first_name` to analyze
- Constructs the URL with that name
- Makes the API call
- Returns a clean dictionary with the prediction results

**Key improvement:** This function is **reusable** with any name. We can call it multiple times with different names without rewriting any code.

### Function 3: Predict Nationality

This function wraps our Nationalize.io API call from Step 3:

In [19]:
def predict_nationality(first_name):
    """
    Predicts the nationality of a person based on their first name using Nationalize.io API.

    Parameters:
        first_name (str): The first name to analyze

    Returns:
        dict: A dictionary containing:
            - 'name': str, the name that was analyzed
            - 'top_country': str or None, the country code with highest probability
            - 'probability': float, the probability of the top prediction
            - 'all_predictions': list, all country predictions with probabilities
    """
    # Construct the API URL with the name parameter
    url = f"https://api.nationalize.io?name={first_name}"

    # Make the request and parse the response
    response = requests.get(url)
    data = response.json()

    # Extract country predictions
    predictions = data['country']

    # Get the top prediction if available
    if len(predictions) > 0:
        top_prediction = predictions[0]
        top_country = top_prediction['country_id']
        probability = top_prediction['probability']
    else:
        top_country = None
        probability = 0

    # Return the prediction data
    return {
        'name': data['name'],
        'top_country': top_country,
        'probability': probability,
        'all_predictions': predictions
    }

# Test the function
print("Testing predict_nationality() function...\n")
nationality_result = predict_nationality(test_user['first_name'])
print(f"Nationality prediction for '{nationality_result['name']}':")
print(f"  Top prediction: {nationality_result['top_country']} ({nationality_result['probability']:.2%})")
print(f"  Actual country: {test_user['country']}")
print(f"\n  All predictions:")
for pred in nationality_result['all_predictions'][:3]:  # Show top 3
    print(f"    {pred['country_id']}: {pred['probability']:.2%}")

Testing predict_nationality() function...

Nationality prediction for 'Simon':
  Top prediction: HU (7.50%)
  Actual country: Denmark

  All predictions:
    HU: 7.50%
    US: 4.28%
    FR: 3.99%


**What this function does:**

- Takes **one parameter:** the `first_name` to analyze
- Handles the case where there might be no predictions (empty list)
- Returns both the top prediction and all predictions for flexibility
- Provides probability information for interpreting confidence

### Function 4: Complete User Analysis

Now let's create a **master function** that combines all three API calls into one complete workflow. This is the power of functions - we can use our smaller functions as building blocks!

In [20]:
def analyze_user():
    """
    Performs a complete user analysis workflow:
    1. Generates a random user
    2. Predicts their age based on first name
    3. Predicts their nationality based on first name
    4. Returns a comparison of predictions vs. actual data

    Returns:
        dict: A comprehensive dictionary containing:
            - 'user': dict, the user information from RandomUser
            - 'age_prediction': dict, age prediction from Agify
            - 'nationality_prediction': dict, nationality prediction from Nationalize
            - 'summary': str, a formatted summary of the analysis
    """
    # Step 1: Get a random user
    user = get_random_user()

    # Step 2: Predict their age using their first name
    age_pred = predict_age(user['first_name'])

    # Step 3: Predict their nationality using their first name
    nat_pred = predict_nationality(user['first_name'])

    # Create a formatted summary string
    summary = f"""
Generated User: {user['first_name']} {user['last_name']}
  Actual age: {user['age']}
  Actual country: {user['country']}

Predictions:
  Age prediction: {age_pred['predicted_age']} (difference: {abs(age_pred['predicted_age'] - user['age']) if age_pred['predicted_age'] else 'N/A'} years)
  Nationality prediction: {nat_pred['top_country']} ({nat_pred['probability']:.1%} confidence)

Accuracy:
  Age accuracy: {'Good' if age_pred['predicted_age'] and abs(age_pred['predicted_age'] - user['age']) <= 10 else 'Poor' if age_pred['predicted_age'] else 'No prediction'}
  Nationality accuracy: {'Exact match!' if nat_pred['top_country'] in user['country'] else 'Different country'}
    """.strip()

    # Return all the data
    return {
        'user': user,
        'age_prediction': age_pred,
        'nationality_prediction': nat_pred,
        'summary': summary
    }

# Test the complete workflow
print("Testing analyze_user() function...\n")
print("="*70)
result = analyze_user()
print(result['summary'])
print("="*70)

Testing analyze_user() function...

Generated User: Fiona Coleman
  Actual age: 55
  Actual country: Ireland

Predictions:
  Age prediction: 48 (difference: 7 years)
  Nationality prediction: CN (26.9% confidence)

Accuracy:
  Age accuracy: Good
  Nationality accuracy: Different country


**What this master function does:**

- **Calls our three helper functions** in sequence
- **Passes data between them:** user's name flows from `get_random_user()` to the prediction functions
- **Combines all results** into one comprehensive output
- **Creates a formatted summary** for easy reading
- **Calculates accuracy metrics** by comparing predictions to actual values

**The power of composition:** Notice how clean this function is. We don't need to repeat any of the API call logic - we just use our building blocks!

### Understanding the Workflow

Here's what happens when you call `analyze_user()`:

```
analyze_user()
    ↓
    calls get_random_user()
        ↓ makes API call to RandomUser
        ↓ returns user dictionary
    ↓
    calls predict_age(first_name)
        ↓ makes API call to Agify
        ↓ returns age prediction
    ↓
    calls predict_nationality(first_name)
        ↓ makes API call to Nationalize
        ↓ returns nationality prediction
    ↓
    combines all results
    ↓
    returns complete analysis
```

**Three API calls, one function!** This is the essence of automation through functions.

### What to Look For

As you review the results, observe:

1. **Age predictions:**
   - Some names will have very accurate predictions (within 5-10 years)
   - Others might be quite far off
   - Some names might return `None` (not in the database)

2. **Nationality predictions:**
   - Common international names often match well
   - Region-specific names might predict nearby countries correctly
   - The probability scores indicate confidence levels

3. **Patterns:**
   - Names from certain regions tend to predict better
   - Very common names (like "John" or "Maria") often have lower accuracy because they're used globally
   - Unique or culturally specific names tend to predict better

### Why the Predictions Aren't Perfect

These APIs use **statistical analysis** based on:
- Historical name databases
- Cultural naming patterns
- Population demographics

They **don't** have access to:
- Current trends or fads
- Individual family histories
- Immigration and cultural mixing
- The actual person's data

This is why predictions are **probabilistic** (based on likelihood) rather than definitive.

---

## What We've Accomplished

Congratulations! You've now built a complete **multi-API integration system**. Let's review what we've learned:

### Core API Concepts

1. ✓ **HTTP Requests:** Using `requests.get()` to send GET requests
2. ✓ **Status Codes:** Understanding 200 (success), 404 (not found), etc.
3. ✓ **Response Parsing:** Converting JSON responses to Python dictionaries
4. ✓ **Query Parameters:** Adding parameters to URLs (`?name=value`)
5. ✓ **Response Formats:** Working with different JSON structures

### Data Structures

1. ✓ **Simple dictionaries:** Like Agify's response with just a few keys
2. ✓ **Nested dictionaries:** Like RandomUser's complex user profiles
3. ✓ **Lists of dictionaries:** Like Nationalize's multiple country predictions
4. ✓ **Chain indexing:** Accessing nested data like `user['name']['first']`

### Programming Skills

1. ✓ **Functions:** Creating reusable code blocks
2. ✓ **Composition:** Building complex functions from simpler ones
3. ✓ **Error handling:** Dealing with None values and empty lists
4. ✓ **Rate limiting:** Being respectful to API servers
5. ✓ **Documentation:** Writing docstrings to explain function behavior

### Real-World Applications

The skills you've learned apply directly to:

- **Data enrichment:** Combining data from multiple sources
- **API integration:** Building systems that use multiple APIs together
- **Automation:** Creating workflows that run without manual intervention
- **Business intelligence:** Gathering and analyzing data from various sources

---

## Additional Tasks: Error Handling

Our current code works well under normal conditions, but what happens when things go wrong? APIs can fail for many reasons:

- **Network issues:** Internet connection problems
- **Server errors:** The API server is down or overloaded
- **Rate limiting:** Too many requests in a short time
- **Invalid requests:** Malformed URLs or parameters

### The Challenge

**Your task:** Improve our functions to handle potential errors gracefully.

### Hint: Using Status Codes

Every HTTP response includes a **status code** that tells you whether the request succeeded:

```python
response = requests.get(url)
status = response.status_code

if status == 200:
    # Success! Process the data
    data = response.json()
else:
    # Something went wrong
    print(f"Error: Received status code {status}")
```

Common status codes:
- **200:** OK - Request succeeded
- **400:** Bad Request - Invalid parameters
- **401:** Unauthorized - Need authentication
- **404:** Not Found - Resource doesn't exist
- **429:** Too Many Requests - Rate limit exceeded
- **500:** Internal Server Error - Server problem

### What to Implement

Consider adding:

1. **Status code checking** before parsing JSON
2. **Error messages** that explain what went wrong
3. **Default values** when APIs fail (e.g., return None)
4. **Retry logic** for temporary failures
5. **Timeout handling** for slow responses

### Example Improved Function

Here's an example of how you might improve the `predict_age()` function:

In [22]:
def predict_age_with_error_handling(first_name):
    """
    Predicts age with proper error handling.

    Parameters:
        first_name (str): The first name to analyze

    Returns:
        dict: Prediction data, or None if the request failed
    """
    try:
        url = f"https://api.agify.io?name={first_name}"
        response = requests.get(url, timeout=10)  # 10 second timeout

        # Check if the request was successful
        if response.status_code == 200:
            data = response.json()
            return {
                'name': data['name'],
                'predicted_age': data['age'],
                'count': data['count'],
                'status': 'success'
            }
        else:
            # Request failed - return error information
            return {
                'name': first_name,
                'predicted_age': None,
                'count': 0,
                'status': f'error',
                'error_code': response.status_code
            }

    except requests.exceptions.Timeout:
        # Request took too long
        return {
            'name': first_name,
            'predicted_age': None,
            'count': 0,
            'status': 'timeout'
        }

    except requests.exceptions.RequestException as e:
        # Some other error occurred (network problem, etc.)
        return {
            'name': first_name,
            'predicted_age': None,
            'count': 0,
            'status': 'connection_error',
            'error_message': str(e)
        }

# Test with a valid name
print("Testing with valid name:")
result = predict_age_with_error_handling("Emma")
print(result)

# Test with an invalid URL (to simulate an error)
print("\nTesting error handling with invalid request:")
# We can't easily simulate API errors in this environment,
# but the error handling is in place for real-world scenarios

Testing with valid name:
{'name': 'Emma', 'predicted_age': 43, 'count': 56913, 'status': 'success'}

Testing error handling with invalid request:


### Your Turn!

Try to:

1. **Add error handling** to the `get_random_user()` function
2. **Add error handling** to the `predict_nationality()` function
3. **Update** the `analyze_user()` function to handle cases where one or more API calls fail
4. **Test** your code by intentionally creating errors (e.g., using an invalid URL)

**Bonus challenges:**

- Add **retry logic:** If a request fails, try again 2-3 times before giving up
- Add **logging:** Print informative messages about what's happening
- Add **validation:** Check if the first_name parameter is valid before making the request
- Create a **summary function** that calculates overall accuracy across multiple users

Good luck! This exercise will help you write more **robust** and **production-ready** code.

---

## Further Learning

Want to expand your API skills? Try these exercises:

### More Practice

1. **Explore other free APIs:**
   - [JSONPlaceholder](https://jsonplaceholder.typicode.com/) - Fake REST API for testing
   - [OpenWeather](https://openweathermap.org/api) - Weather data
   - [REST Countries](https://restcountries.com/) - Country information
   - [The Cat API](https://thecatapi.com/) - Random cat pictures!

2. **Build something new:**
   - Create a weather report generator
   - Build a currency converter using exchange rate APIs
   - Make a quote-of-the-day application

3. **Learn about authentication:**
   - Sign up for an API that requires an API key
   - Learn how to include authentication headers
   - Explore OAuth-based APIs

### Next Steps

- Learn about **POST requests** to send data to APIs
- Explore **asynchronous requests** for faster API calls
- Study **API rate limiting** strategies
- Investigate **web scraping** for sites without APIs
- Learn about **API documentation** standards like OpenAPI/Swagger

The skills you've learned here are fundamental to modern software development and data science!